# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

RANDOM_STATE = 42

# Ищем датасет под тем именем, которое есть рядом с ноутбуком
possible_paths = [
    Path("auto_dataset(2).csv"),
    Path("auto_dataset.csv")
]

DATA_PATH = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Не найден auto_dataset(2).csv или auto_dataset.csv"
    )

data = pd.read_csv(DATA_PATH)

print("Размер датасета:", data.shape)
print("\nТипы признаков:")
print(data.dtypes)

print("\nКоличество пропусков:")
print(data.isna().sum())

data.head()

Размер датасета: (1000, 10)

Типы признаков:
brand                  str
model                  str
vehicleType            str
gearbox                str
fuelType               str
notRepairedDamage      str
powerPS              int64
kilometer            int64
autoAgeMonths        int64
price                int64
dtype: object

Количество пропусков:
brand                0
model                0
vehicleType          0
gearbox              0
fuelType             0
notRepairedDamage    0
powerPS              0
kilometer            0
autoAgeMonths        0
price                0
dtype: int64


,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [3]:
TARGET = "price"

# Отделяем признаки и целевую переменную
X_raw = data.drop(columns=TARGET)
y_raw = data[TARGET].to_numpy(dtype=float)

# Категориальные и числовые признаки
categorical_cols = (
    X_raw
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

numeric_cols = (
    X_raw
    .select_dtypes(exclude=["object", "category"])
    .columns
    .tolist()
)

print("Категориальные признаки:")
print(categorical_cols)

print("\nЧисловые признаки:")
print(numeric_cols)

print("\nРазмер X:", X_raw.shape)
print("Размер y:", y_raw.shape)

Категориальные признаки:
['brand', 'model', 'vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage']

Числовые признаки:
['powerPS', 'kilometer', 'autoAgeMonths']

Размер X: (1000, 9)
Размер y: (1000,)


/var/folders/7_/dsr6ltxx3w9b1jg7jvrmrgd40000gn/T/ipykernel_47523/1053511167.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object", "category"])


3. Разбейте датасет на train val test в отношении 8:1:1

In [4]:

X_train_df, X_temp_df, y_train, y_temp = train_test_split(
    X_raw,
    y_raw,
    test_size=0.2,
    random_state=RANDOM_STATE,
    shuffle=True
)

X_val_df, X_test_df, y_val, y_test = train_test_split(
    X_temp_df,
    y_temp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("Train:", X_train_df.shape)
print("Validation:", X_val_df.shape)
print("Test:", X_test_df.shape)



try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    # Для старых версий sklearn
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            encoder,
            categorical_cols
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_cols
        )
    ],
    sparse_threshold=0
)

X_train = preprocessor.fit_transform(X_train_df)
X_val = preprocessor.transform(X_val_df)
X_test = preprocessor.transform(X_test_df)

X_train = np.asarray(X_train, dtype=float)
X_val = np.asarray(X_val, dtype=float)
X_test = np.asarray(X_test, dtype=float)



X_train = np.column_stack([
    np.ones(X_train.shape[0]),
    X_train
])

X_val = np.column_stack([
    np.ones(X_val.shape[0]),
    X_val
])

X_test = np.column_stack([
    np.ones(X_test.shape[0]),
    X_test
])


y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.reshape(-1, 1)
).ravel()

y_val_scaled = y_scaler.transform(
    y_val.reshape(-1, 1)
).ravel()

y_test_scaled = y_scaler.transform(
    y_test.reshape(-1, 1)
).ravel()

Y_MEAN = float(y_scaler.mean_[0])
Y_SCALE = float(y_scaler.scale_[0])


print("\nПосле preprocessing:")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

Train: (800, 9)
Validation: (100, 9)
Test: (100, 9)

После preprocessing:
X_train: (800, 195)
X_val: (100, 195)
X_test: (100, 195)

y_train: (800,)
y_val: (100,)
y_test: (100,)


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [5]:

STEP_GRID = np.logspace(-5, 0, 6)

MAX_ITER = 5000

def get_eta(
    base_step,
    k,
    lr_mode,
    s0=1.0,
    p=0.5
):

    if lr_mode == "constant":
        return float(base_step)

    if lr_mode == "decay":
        return (
            float(base_step)
            * (s0 / (s0 + k)) ** p
        )

    raise ValueError(
        "lr_mode должен быть 'constant' или 'decay'"
    )


def predict_original_scale(X, w):

    prediction_scaled = X @ w

    prediction_original = (
        Y_MEAN
        + Y_SCALE * prediction_scaled
    )

    return prediction_original


def evaluate_weights(X, y_original, w):

    prediction = predict_original_scale(
        X,
        w
    )

    mse = float(
        np.mean(
            (y_original - prediction) ** 2
        )
    )

    r2 = float(
        r2_score(
            y_original,
            prediction
        )
    )

    return mse, r2


def train_optimizer(
    method,
    lr_mode,
    base_step,
    max_iter=MAX_ITER,
    batch_size=32,
    alpha=0.9,
    beta1=0.9,
    beta2=0.999,
    epsilon=1e-8,
    random_state=RANDOM_STATE
):


    n, d = X_train.shape

    # Начинаем с нулевых весов
    w = np.zeros(d)

    rng = np.random.default_rng(
        random_state
    )


    if method == "SAG":

        # Для каждого train-объекта храним
        # последний вычисленный градиент
        gradient_memory = np.zeros(
            (n, d)
        )

        average_gradient = np.zeros(d)


    elif method == "Momentum":

        # h_0 = 0
        h = np.zeros(d)


    elif method == "Adam":

        # m_0 = 0, v_0 = 0
        m = np.zeros(d)
        v = np.zeros(d)

    best_val_loss, _ = evaluate_weights(
        X_val,
        y_val,
        w
    )

    best_weights = w.copy()
    best_iteration = 0

    val_history = []


    for k in range(max_iter):


        if method in [
            "VGD",
            "Momentum",
            "Adam"
        ]:

            gradient = (
                2.0 / n
            ) * X_train.T @ (
                X_train @ w
                - y_train_scaled
            )

        elif method == "SGD":

            batch_indices = rng.choice(
                n,
                size=min(batch_size, n),
                replace=False
            )

            X_batch = X_train[
                batch_indices
            ]

            y_batch = y_train_scaled[
                batch_indices
            ]

            gradient = (
                2.0 / len(batch_indices)
            ) * X_batch.T @ (
                X_batch @ w
                - y_batch
            )


        elif method == "SAG":

            j = int(
                rng.integers(
                    0,
                    n
                )
            )

            x_j = X_train[j]
            y_j = y_train_scaled[j]

            old_gradient = (
                gradient_memory[j].copy()
            )

            new_gradient = (
                2.0
                * (x_j @ w - y_j)
                * x_j
            )

            gradient_memory[j] = (
                new_gradient
            )

            average_gradient += (
                new_gradient
                - old_gradient
            ) / n

            gradient = (
                average_gradient
            )


        else:

            raise ValueError(
                f"Неизвестный метод: {method}"
            )

        eta = get_eta(
            base_step=base_step,
            k=k,
            lr_mode=lr_mode
        )

        if method in [
            "VGD",
            "SGD",
            "SAG"
        ]:

            w = (
                w
                - eta * gradient
            )


        elif method == "Momentum":

            h = (
                alpha * h
                + eta * gradient
            )

            w = (
                w - h
            )


        elif method == "Adam":

            m = (
                beta1 * m
                + (1.0 - beta1)
                * gradient
            )

            v = (
                beta2 * v
                + (1.0 - beta2)
                * gradient ** 2
            )

            t = k + 1

            m_hat = (
                m
                / (1.0 - beta1 ** t)
            )

            v_hat = (
                v
                / (1.0 - beta2 ** t)
            )

            w = (
                w
                - eta
                * m_hat
                / (
                    np.sqrt(v_hat)
                    + epsilon
                )
            )


        if (
            not np.all(np.isfinite(w))
            or np.linalg.norm(w) > 1e8
        ):
            break

        val_loss, _ = evaluate_weights(
            X_val,
            y_val,
            w
        )

        val_history.append(
            val_loss
        )



        if (
            np.isfinite(val_loss)
            and val_loss < best_val_loss
        ):

            best_val_loss = val_loss

            best_weights = (
                w.copy()
            )

            best_iteration = (
                k + 1
            )


    return {
        "weights": best_weights,
        "best_val_loss": best_val_loss,
        "best_iteration": best_iteration,
        "iterations_done": len(
            val_history
        ),
        "val_history": val_history
    }


def investigate(
    method,
    lr_mode,
    batch_size=32,
    alpha=0.9,
    beta1=0.9,
    beta2=0.999,
    epsilon=1e-8
):
    """
    Перебирает:
        eta     для constant
        lambda  для TimeDecay

    Лучший параметр выбирается ТОЛЬКО
    по минимальному validation loss.

    Test используется только после выбора.
    """

    rows = []
    runs = []

    parameter_name = (
        "eta"
        if lr_mode == "constant"
        else "lambda"
    )

    for base_step in STEP_GRID:

        run = train_optimizer(
            method=method,
            lr_mode=lr_mode,
            base_step=float(base_step),
            max_iter=MAX_ITER,
            batch_size=batch_size,
            alpha=alpha,
            beta1=beta1,
            beta2=beta2,
            epsilon=epsilon,
            random_state=RANDOM_STATE
        )

        train_loss, train_r2 = (
            evaluate_weights(
                X_train,
                y_train,
                run["weights"]
            )
        )

        rows.append({
            parameter_name:
                float(base_step),

            "Loss train":
                train_loss,

            "R2 train":
                train_r2,

            "Loss val":
                run["best_val_loss"],

            "Best iteration":
                run["best_iteration"]
        })

        runs.append(run)


    results = pd.DataFrame(
        rows
    )

    best_position = int(
        np.argmin(
            results[
                "Loss val"
            ].to_numpy()
        )
    )

    best_step = float(
        results.iloc[
            best_position
        ][parameter_name]
    )

    best_run = runs[
        best_position
    ]


    loss_train, r2_train = (
        evaluate_weights(
            X_train,
            y_train,
            best_run["weights"]
        )
    )


    loss_test, r2_test = (
        evaluate_weights(
            X_test,
            y_test,
            best_run["weights"]
        )
    )


    if lr_mode == "constant":

        step_description = (
            f"eta = {best_step:g}"
        )

    else:

        step_description = (
            f"eta_k = {best_step:g} * "
            f"(1 / (1 + k))^0.5"
        )


    summary = {
        "Метод":
            method,

        "Лучший шаг":
            step_description,

        "Loss train":
            loss_train,

        "Loss test":
            loss_test,

        "R2 train":
            r2_train,

        "R2 test":
            r2_test,

        "Число итераций на test":
            best_run[
                "best_iteration"
            ]
    }


    print(
        "=" * 70
    )

    print(
        f"Метод: {method}"
    )

    print(
        "Шаг:",
        (
            "постоянный eta"
            if lr_mode == "constant"
            else "TimeDecay eta(lambda)"
        )
    )

    print(
        "=" * 70
    )

    display(results)

    print(
        f"\nЛучший {parameter_name}: "
        f"{best_step:g}"
    )

    print(
        "Минимальный Loss_val:",
        f"{best_run['best_val_loss']:.4f}"
    )

    print(
        "Лучшая итерация:",
        best_run[
            "best_iteration"
        ]
    )

    print(
        f"Loss_test: {loss_test:.4f}"
    )

    print(
        f"R2_test: {r2_test:.6f}"
    )


    return (
        results,
        summary,
        best_run
    )


(
    vgd_const_results,
    vgd_const_summary,
    vgd_const_best_run
) = investigate(
    method="VGD",
    lr_mode="constant"
)

Метод: VGD
Шаг: постоянный eta


,eta,Loss train,R2 train,Loss val,Best iteration
0,0.00001,5.117356e+07,0.159193,4.669157e+07,5000
1,0.00010,2.352868e+07,0.613412,2.265008e+07,5000
2,0.00100,1.820521e+07,0.700879,2.059038e+07,5000
3,0.01000,1.519322e+07,0.750368,1.825524e+07,5000
4,0.10000,1.276585e+07,0.790251,1.669758e+07,3972
5,1.00000,6.086242e+07,0.000000,5.536406e+07,0



Лучший eta: 0.1
Минимальный Loss_val: 16697582.6607
Лучшая итерация: 3972
Loss_test: 27600146.1175
R2_test: 0.604228


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [6]:
(
    vgd_decay_results,
    vgd_decay_summary,
    vgd_decay_best_run
) = investigate(
    method="VGD",
    lr_mode="decay"
)

Метод: VGD
Шаг: TimeDecay eta(lambda)


,lambda,Loss train,R2 train,Loss val,Best iteration
0,0.00001,6.055166e+07,0.005106,5.508580e+07,5000
1,0.00010,5.786460e+07,0.049256,5.267982e+07,5000
2,0.00100,3.938035e+07,0.352961,3.617655e+07,5000
3,0.01000,2.000885e+07,0.671245,2.090256e+07,3942
4,0.10000,1.707780e+07,0.719403,1.984588e+07,5000
5,1.00000,1.373414e+07,0.774341,1.707195e+07,5000



Лучший lambda: 1
Минимальный Loss_val: 17071953.1224
Лучшая итерация: 5000
Loss_test: 26458719.3574
R2_test: 0.620595


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [7]:
(
    sgd_const_results,
    sgd_const_summary,
    sgd_const_best_run
) = investigate(
    method="SGD",
    lr_mode="constant",
    batch_size=32
)

Метод: SGD
Шаг: постоянный eta


,eta,Loss train,R2 train,Loss val,Best iteration
0,0.00001,5.118561e+07,0.158995,4.670027e+07,5000
1,0.00010,2.351416e+07,0.613651,2.263870e+07,5000
2,0.00100,1.829674e+07,0.699375,2.039511e+07,4468
3,0.01000,1.550400e+07,0.745262,1.741889e+07,4468
4,0.10000,1.411605e+07,0.768066,1.413975e+07,3681
5,1.00000,6.086242e+07,0.000000,5.536406e+07,0



Лучший eta: 0.1
Минимальный Loss_val: 14139746.5667
Лучшая итерация: 3681
Loss_test: 31922991.5816
R2_test: 0.542241


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [8]:
(
    sgd_decay_results,
    sgd_decay_summary,
    sgd_decay_best_run
) = investigate(
    method="SGD",
    lr_mode="decay",
    batch_size=32
)

Метод: SGD
Шаг: TimeDecay eta(lambda)


,lambda,Loss train,R2 train,Loss val,Best iteration
0,0.00001,6.054922e+07,0.005146,5.508346e+07,5000
1,0.00010,5.784169e+07,0.049632,5.265820e+07,5000
2,0.00100,3.926207e+07,0.354904,3.607936e+07,5000
3,0.01000,1.984457e+07,0.673944,2.089089e+07,4468
4,0.10000,1.714086e+07,0.718367,1.960175e+07,4468
5,1.00000,6.086242e+07,0.000000,5.536406e+07,0



Лучший lambda: 0.1
Минимальный Loss_val: 19601752.1589
Лучшая итерация: 4468
Loss_test: 24914197.6299
R2_test: 0.642743


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [9]:
(
    sag_const_results,
    sag_const_summary,
    sag_const_best_run
) = investigate(
    method="SAG",
    lr_mode="constant"
)

Метод: SAG
Шаг: постоянный eta


,eta,Loss train,R2 train,Loss val,Best iteration
0,0.00001,5.250975e+07,0.137239,4.788423e+07,5000
1,0.00010,2.348230e+07,0.614174,2.243031e+07,5000
2,0.00100,1.854194e+07,0.695347,2.089961e+07,4348
3,0.01000,1.615726e+07,0.734528,1.749073e+07,5000
4,0.10000,2.229275e+07,0.633719,2.247546e+07,84
5,1.00000,2.641204e+07,0.566037,2.573981e+07,28



Лучший eta: 0.01
Минимальный Loss_val: 17490728.7838
Лучшая итерация: 5000
Loss_test: 26246735.9618
R2_test: 0.623635


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [10]:
(
    sag_decay_results,
    sag_decay_summary,
    sag_decay_best_run
) = investigate(
    method="SAG",
    lr_mode="decay"
)

Метод: SAG
Шаг: TimeDecay eta(lambda)


,lambda,Loss train,R2 train,Loss val,Best iteration
0,0.00001,6.066254e+07,0.003284,5.518508e+07,5000
1,0.00010,5.890409e+07,0.032176,5.361047e+07,5000
2,0.00100,4.481361e+07,0.263690,4.100018e+07,5000
3,0.01000,2.098069e+07,0.655277,2.100702e+07,3231
4,0.10000,1.861665e+07,0.694119,2.029184e+07,2492
5,1.00000,1.688485e+07,0.722573,1.800099e+07,4329



Лучший lambda: 1
Минимальный Loss_val: 18000987.8360
Лучшая итерация: 4329
Loss_test: 30472697.9189
R2_test: 0.563037


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [11]:
(
    momentum_const_results,
    momentum_const_summary,
    momentum_const_best_run
) = investigate(
    method="Momentum",
    lr_mode="constant",
    alpha=0.9
)

Метод: Momentum
Шаг: постоянный eta


,eta,Loss train,R2 train,Loss val,Best iteration
0,0.00001,2.352580e+07,0.613459,2.264649e+07,5000
1,0.00010,1.820604e+07,0.700866,2.059200e+07,5000
2,0.00100,1.519453e+07,0.750346,1.825630e+07,5000
3,0.01000,1.276917e+07,0.790196,1.669734e+07,3953
4,0.10000,1.280761e+07,0.789565,1.669452e+07,374
5,1.00000,6.086242e+07,0.000000,5.536406e+07,0



Лучший eta: 0.1
Минимальный Loss_val: 16694522.3359
Лучшая итерация: 374
Loss_test: 27556751.5410
R2_test: 0.604850


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [12]:
(
    momentum_decay_results,
    momentum_decay_summary,
    momentum_decay_best_run
) = investigate(
    method="Momentum",
    lr_mode="decay",
    alpha=0.9
)

Метод: Momentum
Шаг: TimeDecay eta(lambda)


,lambda,Loss train,R2 train,Loss val,Best iteration
0,0.00001,5.786683e+07,0.049219,5.268181e+07,5000
1,0.00010,3.937425e+07,0.353061,3.617088e+07,5000
2,0.00100,2.002605e+07,0.670962,2.090689e+07,3845
3,0.01000,1.707753e+07,0.719408,1.984668e+07,5000
4,0.10000,1.373231e+07,0.774371,1.706890e+07,5000
5,1.00000,1.589446e+07,0.738846,1.589507e+07,28



Лучший lambda: 1
Минимальный Loss_val: 15895069.8565
Лучшая итерация: 28
Loss_test: 29617917.9903
R2_test: 0.575294


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [13]:
(
    adam_const_results,
    adam_const_summary,
    adam_const_best_run
) = investigate(
    method="Adam",
    lr_mode="constant",
    beta1=0.9,
    beta2=0.999,
    epsilon=1e-8
)

Метод: Adam
Шаг: постоянный eta


,eta,Loss train,R2 train,Loss val,Best iteration
0,0.00001,4.486920e+07,0.262777,4.061327e+07,5000
1,0.00010,1.468352e+07,0.758742,1.672170e+07,5000
2,0.00100,1.374863e+07,0.774103,1.668610e+07,797
3,0.01000,1.335879e+07,0.780508,1.690462e+07,93
4,0.10000,1.625145e+07,0.732981,1.668905e+07,10
5,1.00000,1.591326e+07,0.738537,1.724252e+07,46



Лучший eta: 0.001
Минимальный Loss_val: 16686097.9423
Лучшая итерация: 797
Loss_test: 26929897.5101
R2_test: 0.613839


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [14]:
(
    adam_decay_results,
    adam_decay_summary,
    adam_decay_best_run
) = investigate(
    method="Adam",
    lr_mode="decay",
    beta1=0.9,
    beta2=0.999,
    epsilon=1e-8
)

Метод: Adam
Шаг: TimeDecay eta(lambda)


,lambda,Loss train,R2 train,Loss val,Best iteration
0,0.00001,6.033426e+07,0.008678,5.486127e+07,5000
1,0.00010,5.578813e+07,0.083373,5.061315e+07,5000
2,0.00100,2.769551e+07,0.544949,2.577876e+07,5000
3,0.01000,1.395826e+07,0.770659,1.660866e+07,1004
4,0.10000,1.444670e+07,0.762634,1.649871e+07,18
5,1.00000,1.371410e+07,0.774670,1.705331e+07,49



Лучший lambda: 0.1
Минимальный Loss_val: 16498707.7180
Лучшая итерация: 18
Loss_test: 28494362.5817
R2_test: 0.591405


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [16]:
comparison = pd.DataFrame([
    vgd_const_summary,
    vgd_decay_summary,

    sgd_const_summary,
    sgd_decay_summary,

    sag_const_summary,
    sag_decay_summary,

    momentum_const_summary,
    momentum_decay_summary,

    adam_const_summary,
    adam_decay_summary
])

comparison

,Метод,Лучший шаг,Loss train,Loss test,R2 train,R2 test,Число итераций на test
0,VGD,eta = 0.1,1.276585e+07,2.760015e+07,0.790251,0.604228,3972
1,VGD,eta_k = 1 * (1 / (1 + k))^0.5,1.373414e+07,2.645872e+07,0.774341,0.620595,5000
2,SGD,eta = 0.1,1.411605e+07,3.192299e+07,0.768066,0.542241,3681
3,SGD,eta_k = 0.1 * (1 / (1 + k))^0.5,1.714086e+07,2.491420e+07,0.718367,0.642743,4468
4,SAG,eta = 0.01,1.615726e+07,2.624674e+07,0.734528,0.623635,5000
5,SAG,eta_k = 1 * (1 / (1 + k))^0.5,1.688485e+07,3.047270e+07,0.722573,0.563037,4329
6,Momentum,eta = 0.1,1.280761e+07,2.755675e+07,0.789565,0.604850,374
7,Momentum,eta_k = 1 * (1 / (1 + k))^0.5,1.589446e+07,2.961792e+07,0.738846,0.575294,28
8,Adam,eta = 0.001,1.374863e+07,2.692990e+07,0.774103,0.613839,797
9,Adam,eta_k = 0.1 * (1 / (1 + k))^0.5,1.444670e+07,2.849436e+07,0.762634,0.591405,18


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

In [2]:
#1) По данной таблице я сделала вывод, что наилучшим вариантом будет SGD c уменьшающимся шагом. Такой вывод я сделала, потому что смотрела на значение Loss train в таблице и поняла, что низкие значения у SGD, Adam и Momentum. Далее стала смотреть на Loss test, так как то, что  у нас хороший результат на train не значит что на test он тоже будет такой. Тут loss уже больше. но хорошее значение у SGD c уменьшающимся шагом. У остальных значения уже выше, но не будем делать поспешных выводов. Также проанализируем столбики с R. Тут уже чем ближе к 1, тем лучше. Тут тоже самое высокое значение у SGD c уменьшающимся шагом. Таким образом я и выбрала SGD с уменьшающимся шаком. Еще добавлю что последняя колонка показывает на какой итерации была получена минимальная ошибка, но по этому параметру нельзя сказать, что самое меньшее значение автоматически самая лучшая моделька, так как у модельки с наименьшим значением "Число итераций на test" (использовали Adam + уменьш шаг), но результаты не самые лучшие по другим колонкам.

In [ ]:
#2) Сначала думаю стоит объяснить просто значение R^2, Я понимаю это как коэффициент, который говорит нам насколько наше значение модели лучше чем просто обычное среднее значение. По сути сравниваем с очень простой моделью, которая ничего не знает об автомобилях и просто всегда ставит ср. цену. Если R^2=1, то это идеальное попадание в реальные значения, если R^2=0, то это ничем не лучше чем среднее, если R^2<1, то еще хуже чем среднее. В нашей таблице разница в том, на каких данных посчитана R^2 - на train или test. Важно смотреть на разницу между этими двумя значениями. Оба высокие - круто, модель хорошо предсказывает как на трейн так и на тест. у нас для SGD (не фиксированный шаг) значение на тест ниже, что в целом имеет место быть. Может тогда сильно подстроиться под обучающие данные или выборки заметно отличаются.

In [ ]:
#R^2 test как раз показывает, сохранилось ли качество модели на данных, которые она не использовала при обучении (по сути не видела их). Чем выше значение, тем лучше метод работает на новых данных. Обязательно нужно смотреть на разницу между трейн и тест, если она небольшая, значит качество модели переносится на новые хорошо, если разница большая, значит модель намного лучше работает на обучающих данных, чем на новых. Возможно переобучение